### import & path

In [2]:
from pathlib import Path 
from collections import Counter 
from PIL import Image, ImageOps
from IPython.display import display

import hashlib
import random
import math 

import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
import matplotlib.patches as patches

In [3]:
SEED = 42 

random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path(
    r"D:\project\pill-object-detection"
)

DATASET_ROOT = (
    PROJECT_ROOT / "pill-dataset"
)

IMAGE_ROOT = (
    DATASET_ROOT / "images"
)

LABEL_ROOT = (
    IMAGE_ROOT / "labels"
)

IMAGE_EXTS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp"
}

SPLITS = ["train", "val"]

print("Dataset:", DATASET_ROOT)

Dataset: D:\project\pill-object-detection\pill-dataset


### จำนวน Image & Labels

In [4]:
rows = []

for split in SPLITS:
    image_dir = IMAGE_ROOT / split
    label_dir = LABEL_ROOT / split

    #ค้นหารูป ที่นามสกุลตรง IMAGE_EXTs
    images = [
        p 
        for p in image_dir.iterdir()
        if p.suffix.lower() in IMAGE_EXTS
    ]

    labels = list(
        label_dir.glob("*.txt")
    )

    # ดึงเฉพาะชื่อไฟล์ ไม่เอา นามสกุลไฟล์
    image_steams = {
        p.stem.lower()
        for p in images
    }

    label_stems = {
        p.stem.lower()
        for p in labels 
    }

    rows.append({
        "split": split,
        "images": len(images),
        "labels": len(labels),
        "missimg_labels": len(image_steams - label_stems), #จำนวนภาพไม่มี label
        "orphan_labels": len(label_stems - image_steams), # จำนวนไฟล์ label ไม่มีคู่
    })

# แปลง rows ให้เป็น DataFrame
pairing_df = pd.DataFrame(rows)
display(pairing_df)

,split,images,labels,missimg_labels,orphan_labels
0,train,10559,0,10559,0
1,val,1508,0,1508,0


### image metadata

In [ ]:
image_rows = []
corrupt_images = []

for splot in SPLITS:
    image_dir = IMAGE_ROOT / split
    label_dir = LABEL_ROOT / split

    # วน loop อ่านทีละไฟล์ใน Folder รูป
    for image_path in image_dir.iterdir():
        if image_path.suffix.lower() not in IMAGE_EXTS:
            continue

        # สร้าง path หาไฟล์ ชื่อเดียวกับรูป
        label_path = (
            label_dir / f"{image_path.stem}.txt"
        )

        try:
            with Image.open(image_path) as im:
                width, height = im.size 
                # ตรวจสอบความสมบูรณ์ ไม่มีความเสียหาย
                im.verify()

            image_rows.append({
                "split": split,
                "stem": image_path.stem, # ชื่อไฟล์ 
                "image_path": image_path, 
                "label_path": label_path,
                "width": width,
                "height": height, 
                "aspect_ration": ( width / height),
            })
        except Exception as e:
            corrupt_images.append({
                "path": image_path,
                "error": str(e),
            })

images_df = pd.DataFrame(
    image_rows
)

print("Images:", len(images_df))
print("Corrupt:", len(corrupt_images))

display(images_df.head())

Images: 3016
Corrupt: 0


,split,stem,image_path,label_path,width,height,aspect_ration
0,val,Negatives_I0006,D:\project\pill-object-detection\pill-dataset\...,D:\project\pill-object-detection\pill-dataset\...,720,720,1.0
1,val,Negatives_I0014,D:\project\pill-object-detection\pill-dataset\...,D:\project\pill-object-detection\pill-dataset\...,720,720,1.0
2,val,Negatives_I0022,D:\project\pill-object-detection\pill-dataset\...,D:\project\pill-object-detection\pill-dataset\...,720,720,1.0
3,val,Negatives_I0030,D:\project\pill-object-detection\pill-dataset\...,D:\project\pill-object-detection\pill-dataset\...,720,720,1.0
4,val,Negatives_I0038,D:\project\pill-object-detection\pill-dataset\...,D:\project\pill-object-detection\pill-dataset\...,720,720,1.0
